# 02. Grass Court Analysis
Wimbledon-specific patterns in the data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_parquet('../data/processed/matches_clean.parquet')
grass = df[df['surface'] == 'Grass'].copy()

print(f'Total matches: {len(df):,}  |  Grass: {len(grass):,}')

## Serve dominance by surface

In [ ]:
surfaces_of_interest = ['Hard', 'Clay', 'Grass']
plot_df = df[df['surface'].isin(surfaces_of_interest)].dropna(subset=['w_serve_pct'])

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=plot_df,
    x='surface',
    y='w_serve_pct',
    order=surfaces_of_interest,
    ax=ax,
    palette={'Hard': 'steelblue', 'Clay': 'sienna', 'Grass': 'seagreen'},
    width=0.5
)
ax.set_xlabel('Surface')
ax.set_ylabel('Winner serve point win %')
ax.set_title('Winner Serve Dominance by Surface')
import matplotlib.ticker as mticker
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

## Ace rate on grass

In [ ]:
ace_data = {
    surf: df[df['surface'] == surf]['w_ace_rate'].dropna().values
    for surf in ['Grass', 'Hard', 'Clay']
}

colors = {'Grass': 'seagreen', 'Hard': 'steelblue', 'Clay': 'sienna'}

fig, ax = plt.subplots(figsize=(9, 4))
for surf, values in ace_data.items():
    ax.hist(values, bins=50, alpha=0.5, label=f'{surf} (mean={values.mean():.3f})',
            color=colors[surf], density=True)

ax.set_xlabel('Ace rate (aces / service points)')
ax.set_ylabel('Density')
ax.set_title('Winner Ace Rate Distribution by Surface')
ax.legend()
plt.tight_layout()
plt.show()

## Wimbledon-specific trends

In [ ]:
wimbledon = df[df['tourney_name'].str.contains('Wimbledon', case=False, na=False)].copy()
wimbledon['year'] = wimbledon['tourney_date'].dt.year

yearly_stats = wimbledon.groupby('year').agg(
    mean_winner_serve=('w_serve_pct', 'mean'),
    mean_winner_ace_rate=('w_ace_rate', 'mean'),
    mean_winner_1st_won=('w_1st_won_pct', 'mean'),
    n_matches=('match_num', 'count')
).dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, col, title, fmt in zip(
    axes,
    ['mean_winner_serve', 'mean_winner_ace_rate', 'mean_winner_1st_won'],
    ['Serve win %', 'Ace rate', '1st serve won %'],
    ['%.2f', '%.3f', '%.2f']
):
    ax.plot(yearly_stats.index, yearly_stats[col], marker='o', color='seagreen')
    ax.set_title(f'Wimbledon: {title}')
    ax.set_xlabel('Year')

plt.suptitle('Wimbledon Serve Trends Over Time', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f'Wimbledon matches in dataset: {len(wimbledon):,}')

## Top grass court players

In [ ]:
grass_2010 = grass[grass['tourney_date'].dt.year >= 2010]

grass_wins = grass_2010['winner_name'].value_counts().rename('wins')
grass_losses = grass_2010['loser_name'].value_counts().rename('losses')

grass_stats = pd.concat([grass_wins, grass_losses], axis=1).fillna(0)
grass_stats['total'] = grass_stats['wins'] + grass_stats['losses']
grass_stats['win_rate'] = grass_stats['wins'] / grass_stats['total']

top_grass = (
    grass_stats[grass_stats['total'] >= 20]
    .nlargest(15, 'win_rate')
    .sort_values('win_rate')
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_grass.index, top_grass['win_rate'], color='seagreen')
ax.set_xlabel('Grass court win rate')
ax.set_title('Top 15 Grass Court Players by Win Rate (min 20 matches, 2010+)')
import matplotlib.ticker as mticker
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
for i, (name, row) in enumerate(top_grass.iterrows()):
    ax.text(row['win_rate'] + 0.005, i, f"{row['win_rate']:.1%} ({int(row['total'])})",
            va='center', fontsize=8)
plt.tight_layout()
plt.show()